# Module 9 — Capstone: The Whole Pipeline

Part of the RAG Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

**Where this module starts**
- RAG (retrieval-augmented generation) answers a question in two steps:
  1. A search program finds the few texts in a collection that match the question.
  2. A language model, the kind of program behind chat assistants, reads those texts and writes the answer.
- A RAG system is built from five parts:
  1. **Chunk**: split each document into short pieces, called chunks
  2. **Embed**: turn each chunk into an embedding, a list of numbers that an embedding model makes from a text, so that texts with similar meaning get similar embeddings
  3. **Store**: keep every chunk's embedding in one table
  4. **Retrieve**: for each question, return the chunks that match it best
  5. **Generate**: the language model reads those chunks and writes the answer
- Everything the language model gets, the question included, is called the prompt. In part 5, the prompt holds the returned chunks, each with a number, then the question, then instructions for the answer.

**This module's point:** it builds no new part. It runs the whole system, all five parts, on new questions, ones this playbook has not answered before, then changes one setting at a time and runs it again, to find which setting moves the results most. It ends with a guide: for a new problem, which tool fits it, RAG or another one.

**The original system**

The original system this module starts from has these settings:
- Part 1, chunk: each news article of the collection is split into its paragraphs, one chunk per paragraph.
- Part 2, embed: an existing embedding model, nomic-embed-text.
- Part 3, store: has no setting to choose.
- Part 4, retrieve: hybrid search, which combines a meaning search (it compares the question's embedding with each chunk's embedding) and a keyword search (it scores the words a chunk shares with the question). Each chunk gets a hybrid score = 1 / (60 + its rank in the meaning search) + 1 / (60 + its rank in the keyword search), where rank 1 is the best match and 60 is a standard setting. The search returns the chunks with the highest scores while their words stay at or below 400 in total, a number chosen for this playbook, and always at least 1 chunk.
- Part 5, generate: the language model llama3.2:3b, a small, free model that runs on this computer. Its instructions ask for one or two sentences, and set three rules:
  1. Use only the chunks.
  2. After each sentence of the answer, write the numbers of the chunks it comes from in square brackets, such as [2]. This is called citing.
  3. If the chunks do not hold the answer, reply only *"Not enough information."*

**The five changes**

Each change below alters one setting of the original system and keeps the rest. The original system and the five changes make six versions:

| change | part | the original system's setting | the changed setting | taught in |
|---|---|---|---|---|
| meaning search | 4, retrieve | hybrid search | meaning search alone | Modules 3 and 4 |
| keyword search | 4, retrieve | hybrid search | keyword search alone | Modules 3 and 4 |
| sentence chunks | 1, chunk | one chunk per paragraph | one chunk per sentence | Module 5 |
| 1,600 words | 4, retrieve | at most 400 words returned | at most 1,600 words returned, four times as many, a number chosen for this module | Module 5 |
| prompt without rules | 5, generate | the three rules | no rules: the instructions ask only for one or two sentences | Module 6 |

Part 2, embed, keeps its one embedding model, and part 3, store, has no setting, so neither is changed in this module.

**What this module measures**

The test questions come from a public dataset called MultiHop-RAG, of questions about news articles. Each question with an answer was written from two to four articles. For each such question, the dataset gives the correct answer and lists the sentences in those articles that support it, called the listed sentences. The dataset has four kinds of question:
1. Inference: the correct answer is the name of a company or person that the articles describe.
2. Comparison: the correct answer is yes or no, about whether two or more articles say the same thing.
3. Temporal: the correct answer is yes or no, about whether reports published on different dates agree.
4. Null: no article holds the answer, and the correct answer is *"Insufficient information."* An answer to a null question is right when it declines, in any words.

The original system and each change get two grades:
1. A search grade: the share of a question's listed sentences that the returned chunks hold, averaged over the questions. It grades the returned chunks alone, so it needs no answers, and it runs on several hundred questions.
2. Right answers: how many answers are *right*, ruled by a hand check, a reading of each answer next to the dataset's correct answer. It grades the whole system, on fewer questions, because each answer takes the language model several seconds to write.

<u>Note</u>: a right answer is not always grounded, that is, taken from the returned chunks. It can be a lucky guess, or come from the text the language model learned from in training. Such an answer counts as right although the chunks did not give it. **[only in this module]** The hand check does not split right answers into grounded and not grounded, because that means reading each claim of all 240 answers against their returned chunks.

**Two tags in this module**
- **[in real life]** marks a step that happens in a real system: in a test before the system goes live, where the analyst runs test questions that come with a correct answer, or when a live user asks a question.
- **[only in this module]** marks a step that a real system would not do, chosen to make this module simpler or faster.

**[in real life]** A person makes the hand check. **[only in this module]** A larger language model made the hand check, by reading each answer, and no person has checked its rulings.

The module runs four steps:
1. The search grade: the original system and each change search all the new questions except the null ones, and the share of listed sentences returned is compared.
2. Right answers: the six versions answer only a sample of the new questions, the same 40 for each version, 10 of each kind, and the hand check counts the right answers.
3. One table: each change's gap from the original system in Steps 1 and 2, side by side, with the time each answer takes, to see which setting moved the results most.
4. The guide: kinds of problem, each with the tool that fits it.

**What this module produces**
1. The search grade of the original system and of each change.
2. The count of right answers of the original system and of each change, out of 40.
3. One table of every change's gap from the original system in both grades, and its time per answer.
4. A guide from the shape of a problem to the right tool.

## From the curriculum

> ## <u>Module 9 — Capstone: The Whole Pipeline</u>
>
> **Concepts**
> - The full pipeline end to end on new questions
> - Which setting moved the results most
> - A guide from the problem's shape to the right tool
>
> **Worked example:** the full pipeline run, graded and ruled on, with every setting from
> Modules 3–8 compared in one table.

In this module, the full pipeline is the original system, the new questions are questions no earlier module of this playbook wrote an answer to, "graded" is the search grade, and "ruled on" is the hand check of the answers. Modules 7 and 8 changed no setting of the original system: they changed how answers are checked and which questions are asked, so this module uses their hand check and their split by kind of question. The five changes come from earlier modules: the ways of searching from Modules 3 and 4, sentence chunks and the words returned from Module 5, and the prompt rules from Module 6. The worked example is Steps 1 to 3.

The code below loads the tools and sets the two prompts: the original system's prompt, with the three rules, and the prompt without rules. `common` is this playbook's shared helper file: it loads the data, calls the models, and prints tables.

In [1]:
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import common
from common import show

K1, B, RRF = 1.2, 0.75, 60                         # standard settings, not chosen here: K1 and B of the keyword search (they limit how much repeated words and long chunks raise its score), and RRF (reciprocal rank fusion, the name of the hybrid score), its 60
SAMPLE = 10                                        # answered questions of each kind, chosen for run time
MAX_TOKENS = 120                                   # longest answer, in tokens (word-pieces, about 3/4 of a word each), chosen for run time
KINDS = ["inference", "comparison", "temporal", "null"]
tidy = lambda text: " ".join(text.split())       # one space between words, no line breaks

PROMPTS = {
    "rules": ("Here are chunks of news articles:\n\n{chunks}\n\nQuestion: {question}\n"
              "Answer in one or two sentences, using only the chunks above. "
              "After each sentence of your answer, write the numbers of the chunks it comes from in square brackets, such as [2]. "
              "If the chunks do not hold the answer, reply only: Not enough information."),
    "no rules": ("Here are chunks of news articles:\n\n{chunks}\n\nQuestion: {question}\n"
                 "Answer in one or two sentences.")}
numbered = lambda texts: "\n\n".join(f"[{i}] {t}" for i, t in enumerate(texts, start=1))

## The data: the collection and the new questions

The dataset is MultiHop-RAG (Tang and Yang, 2024):
- 609 news articles, published between September and December 2023, each labelled with one category, such as technology or sports
- 2,556 questions about those articles, of the four kinds listed at the top

**[only in this module]** The collection is the 172 technology articles, not all 609: making the embeddings of all 609 articles' chunks would take hours on this computer.

The test questions are of the four kinds. Inference, comparison and temporal questions are used only when all their listed sentences come from technology articles, and comparison and temporal questions only when their correct answer is the word yes or no: a few have another correct answer, and are left out so that every answer can be ruled as a yes or a no. Null questions are drawn from all null questions: a null question has no answer in any of the 609 articles, so none in the technology articles either. Most null questions are about other topics than technology. Whatever its topic, a null question tests one thing: whether the system declines when the collection holds no answer.

**[only in this module]** The questions are new: each earlier module of this playbook that wrote answers drew its questions at random with a fixed seed, a starting number that makes the same draw come back on every run. The code below repeats those draws and leaves their questions out. A question this playbook has already answered could have shaped a choice made along the way, so a grade on it could look better than the system would do on questions it has not seen.

From the new questions:
1. Step 1, the search grade, uses all of them except the null questions (648 questions: every new inference, comparison and temporal question). Null questions list no sentences, so they have no search grade.
2. Step 2, right answers, uses only a sample of them (40 questions: 10 new questions of each kind, the null kind included), drawn at random with the same fixed seed. 10 per kind is a number chosen for run time: each of the 40 questions is answered 6 times, by the original system and by each of the five changes, so Step 2 writes 240 answers. 10 questions per kind is small: on another 10 drawn the same way, each count in Step 2 could differ.

In [2]:
corpus = common.load_corpus()
questions = common.load_questions()
articles = corpus[corpus["category"] == "technology"].reset_index(drop=True)
titles = set(articles["title"])

in_technology = questions["evidence_list"].map(lambda evidence: bool(evidence) and {e["title"] for e in evidence} <= titles)
yes_or_no = questions["answer"].str.lower().isin(["yes", "no"])
kind_is = lambda name: questions["question_type"] == name + "_query"
pools = {
    "inference": questions[in_technology & kind_is("inference")],
    "comparison": questions[in_technology & kind_is("comparison") & yes_or_no],
    "temporal": questions[in_technology & kind_is("temporal") & yes_or_no],
    "null": questions[kind_is("null")],
}

# The earlier draws, repeated exactly: (module, the pool it drew from, questions drawn).
EARLIER = [
    ("Module 6", pools["inference"], 40), ("Module 6", pools["null"], 40),
    ("Module 7", pools["inference"], 30),
    ("Module 7", questions[in_technology & (kind_is("comparison") | kind_is("temporal")) & yes_or_no], 30),
    *[("Module 8", pools[kind], 20) for kind in KINDS],
]
answered_before = set()
for _, pool, size in EARLIER:
    answered_before |= set(pool.sample(size, random_state=common.SEED).index)
new = {kind: pool[~pool.index.isin(answered_before)] for kind, pool in pools.items()}
graded = pd.concat([new[kind] for kind in ["inference", "comparison", "temporal"]])
drawn = {kind: table.sample(SAMPLE, random_state=common.SEED) for kind, table in new.items()}

print(f"Technology articles, the collection = {len(articles):,}")
print()
print("Questions by kind:")
print()
rows = [{"kind": k, "questions that can be drawn": f"{len(pools[k]):,}", "answered before": str(len(pools[k]) - len(new[k])),
         "new": f"{len(new[k]):,}", "Step 1: search grade": f"{len(new[k]):,}" if k != "null" else "—",
         "Step 2: answered": str(SAMPLE)} for k in KINDS]
rows.append({"kind": "total", **{c: f"{sum(int(r[c].replace(',', '')) for r in rows if r[c] != '—'):,}"
                                 for c in ["questions that can be drawn", "answered before", "new", "Step 1: search grade", "Step 2: answered"]}})
show(pd.DataFrame(rows))

Technology articles, the collection = 172

Questions by kind:

kind        questions that can be drawn  answered before  new  Step 1: search grade  Step 2: answered
inference                           371               40  331                   331                10
comparison                          191               34  157                   157                10
temporal                            194               34  160                   160                10
null                                301               40  261                     —                10
total                             1,057              148  909                   648                40


<u>Note</u>: the earlier draws overlap. The same seed shuffles the same questions into the same order, and each draw takes the first ones, so a smaller draw from the same questions is contained in a larger one.

**[in real life]** The code below builds the system's parts 1 to 4 twice, once on paragraph chunks and once on sentence chunks, and sets the six versions: the original system and the five changes. A sentence ends at ".", "!" or "?" followed by a space and a capital letter or an opening quote. `common` saves every embedding in a file on this computer, so a re-run reads them back instead of making them again.

In [3]:
words_in = lambda text: re.findall(r"[a-z0-9]+", text.lower())
flat = lambda text: " ".join(text.split()).lower()


def paragraphs(body):
    return [tidy(p) for p in body.split("\n\n") if p.strip()]


def sentences(body):
    return [s for p in paragraphs(body) for s in re.split(r'(?<=[.!?])\s+(?=[A-Z"“])', p)]


def keyword_search(chunks):
    # BM25 (Best Matching 25), the keyword search's score. Returns a function: question -> one score per chunk.
    tokens = [words_in(c) for c in chunks]
    n = len(chunks)
    length = np.array([len(t) for t in tokens])
    holding = defaultdict(lambda: ([], []))      # word -> (chunks that hold it, times in each)
    for i, words in enumerate(tokens):
        for w, times in Counter(words).items():
            holding[w][0].append(i)
            holding[w][1].append(times)
    holding = {w: (np.array(rows), np.array(times, dtype=float)) for w, (rows, times) in holding.items()}
    length_part = K1 * (1 - B + B * length / length.mean())

    def scores(question):
        total = np.zeros(n)
        for w in dict.fromkeys(words_in(question)):
            if w in holding:
                rows, times = holding[w]
                rarity_weight = np.log(1 + (n - len(rows) + 0.5) / (len(rows) + 0.5))
                total[rows] += rarity_weight * (times * (K1 + 1)) / (times + length_part[rows])
        return total
    return scores


def ranks_of(scores):
    # Rank 1 = highest score. Ties keep the collection's order.
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


unit = lambda m: m / np.linalg.norm(m, axis=1, keepdims=True)
index = {}
for way, split in [("paragraph", paragraphs), ("sentence", sentences)]:
    chunks = [c for body in articles["body"] for c in split(body)]
    index[way] = {"chunks": chunks, "flat": [flat(c) for c in chunks],
                  "words": np.array([len(c.split()) for c in chunks]),
                  "store": unit(common.embed(chunks, prefix="search_document: ")),   # nomic-embed-text expects this prefix before a passage, and "search_query: " before a question
                  "keyword": keyword_search(chunks)}
    print(f"Chunks, one per {way} = {len(chunks):,}")

SYSTEMS = {  # version -> (way of splitting, way of searching, words returned, prompt)
    "the original system":  ("paragraph", "hybrid",  400,  "rules"),
    "meaning search":       ("paragraph", "meaning", 400,  "rules"),
    "keyword search":       ("paragraph", "keyword", 400,  "rules"),
    "sentence chunks":      ("sentence",  "hybrid",  400,  "rules"),
    "1,600 words":          ("paragraph", "hybrid",  1600, "rules"),
    "prompt without rules": ("paragraph", "hybrid",  400,  "no rules"),
}
PART = {"meaning search": "4, retrieve", "keyword search": "4, retrieve", "sentence chunks": "1, chunk",
        "1,600 words": "4, retrieve", "prompt without rules": "5, generate"}


def retrieve(question, way, search, budget):
    # The positions of the chunks returned for a question, best first: up to `budget` words in total, and at least 1 chunk.
    meaning_ranks = ranks_of(index[way]["store"] @ unit(common.embed([question], prefix="search_query: "))[0])
    keyword_ranks = ranks_of(index[way]["keyword"](question))
    rank = {"meaning": meaning_ranks, "keyword": keyword_ranks}.get(search)
    score = 1 / (RRF + meaning_ranks) + 1 / (RRF + keyword_ranks) if search == "hybrid" else -rank.astype(float)
    order = np.argsort(-score, kind="stable")
    k = max(1, int((np.cumsum(index[way]["words"][order]) <= budget).sum()))
    return list(order[:k])

Chunks, one per paragraph = 7,137


Chunks, one per sentence = 16,431


## Step 1 of 4: the search grade

**Which setting changes how many listed sentences the search returns?**

The search grade is recall: the share of a question's listed sentences that the returned chunks hold, averaged over the questions.

**formula**:

Recall of a question = (listed sentences held whole by a returned chunk) / (sentences listed for the question)

- Recall is 1 when the returned chunks hold every listed sentence, and 0 when they hold none.
- A listed sentence counts when one returned chunk holds it word for word.
- The search's recall is the average of the questions' recalls. Each question counts once, whether it lists few sentences or many.

Recall grades the returned chunks, so it needs no answers. The change to part 5, the prompt without rules, returns the original system's chunks, so it gets the original system's recall.

**Is a gap from the original system real?** The questions graded here are one set of questions. Another set written the same way would give somewhat different recalls. Every change is graded on the same questions as the original system, so each change is compared with the original system question by question:

- Difference on a question = (the change's recall on the question) − (the original system's recall on the question)
- Gap = the average of the differences over all the questions

**formula**:

Standard error of the gap = (standard deviation of the differences) / √(number of questions)

95% interval = gap − (1.96 × standard error) to gap + (1.96 × standard error)

In words:
- The standard deviation of the differences says how far the differences on single questions typically are from the gap.
- The standard error says how far the gap would typically move from one set of test questions to another of the same size. The standard error, not the standard deviation, goes into the interval, because the interval is about the gap, a number calculated from a sample of questions, not about one question's difference.
- 1.96 is the standard value for a 95% interval: over many sets of test questions, intervals made this way hold the true gap 95% of the time.
- The pass rule, the standard one: the gap is real when 0 is outside the interval. When 0 is inside, the data cannot tell the change from the original system.

**[in real life]** In a test before the system goes live, the analyst searches every test question with the original system and with each change, and calculates recall from the returned chunks.

In [4]:
def recall(row, way, search, budget):
    returned = [index[way]["flat"][i] for i in retrieve(row["query"], way, search, budget)]
    listed = [flat(e["fact"]) for e in row["evidence_list"]]
    return np.mean([any(s in chunk for chunk in returned) for s in listed])


recalls = {name: np.array([recall(row, way, search, budget) for _, row in graded.iterrows()])
           for name, (way, search, budget, _) in SYSTEMS.items()}


def gap_and_interval(differences, widen=1):
    # widen=2 doubles the half-width, 1.96 × standard error, on each side of the gap
    d = np.asarray(differences, dtype=float)
    standard_error = d.std(ddof=1) / np.sqrt(len(d))
    return d.mean(), d.mean() - widen * 1.96 * standard_error, d.mean() + widen * 1.96 * standard_error


sign = lambda v: f"{v:+.3f}".replace("-", "−")
rows = []
for name, (way, search, budget, _) in SYSTEMS.items():
    gap, low, high = gap_and_interval(recalls[name] - recalls["the original system"])
    _, low2, high2 = gap_and_interval(recalls[name] - recalls["the original system"], widen=2)
    same = name in ("the original system", "prompt without rules")
    rows.append({"version": name, "words returned": f"{budget:,}", "recall": f"{recalls[name].mean():.3f}",
                 "gap from the original system": "—" if same else sign(gap),
                 "95% interval of the gap": "—" if same else f"{sign(low)} to {sign(high)}",
                 "real gap": "—" if same else ("yes" if low > 0 or high < 0 else "no"),
                 "doubled interval of the gap": "—" if same else f"{sign(low2)} to {sign(high2)}",
                 "real gap: doubled interval": "—" if same else ("yes" if low2 > 0 or high2 < 0 else "no")})
listed_count = Counter(flat(e["fact"]) for evidence in graded["evidence_list"] for e in evidence)
sharing = sum(any(listed_count[flat(e["fact"])] > 1 for e in evidence) for evidence in graded["evidence_list"])
print(f"Questions that share a listed sentence with another question = {sharing} of {len(graded)}")
print()
print(f"Recall of the returned chunks, {len(graded)} new questions:")
print()
show(pd.DataFrame(rows))

Questions that share a listed sentence with another question = 638 of 648

Recall of the returned chunks, 648 new questions:

version               words returned  recall  gap from the original system  95% interval of the gap  real gap  doubled interval of the gap  real gap: doubled interval
the original system              400   0.381                             —  —                        —         —                            —
meaning search                   400   0.288                        −0.093  −0.110 to −0.076         yes       −0.127 to −0.060             yes
keyword search                   400   0.424                        +0.043  +0.026 to +0.059         yes       +0.010 to +0.076             yes
sentence chunks                  400   0.455                        +0.074  +0.053 to +0.095         yes       +0.032 to +0.116             yes
1,600 words                    1,600   0.635                        +0.254  +0.234 to +0.273         yes       +0.214 to +0.293      

**Observation:** every change to part 1 or part 4 moves recall by more than chance: 0 is outside every interval. The questions share listed sentences, as the first line of the output shows, so their recalls rise and fall together, and the 95% intervals are narrower than they should be. As a stress test, the last two columns double each interval's half-width: doubled interval = gap − (2 × 1.96 × standard error) to gap + (2 × 1.96 × standard error). The factor 2 is chosen for this module, and if the questions were unrelated, the doubled interval would hold the true gap more than 99.9% of the time. 0 stays outside every doubled interval too. The largest gap is the words returned: at 1,600 words, recall is 0.254 higher than at 400 words, because the search returns four times as much text. The other changes keep the limit of at most 400 words, so their gaps compare about the same amount of text.

**Observation:** with the same limit of 400 words, sentence chunks return the most listed sentences (+0.074), then keyword search alone (+0.043). Meaning search alone returns the fewest (−0.093). So on these questions, hybrid search returns fewer listed sentences than keyword search alone.

**Hypothesis:** these questions name companies, people and news sites, words that the listed sentences repeat, so keyword search ranks those sentences high. Meaning search ranks them lower, and hybrid search, which adds a score from each ranking, is pulled down by meaning search's ranks. This module does not test the hypothesis.

**Observation:** even at 1,600 words, the search misses about a third of the listed sentences: 1 − 0.635 = 0.365.

**Conclusion:** returning more words moves the search grade most. With the same limit of 400 words, splitting the articles into sentences moves it most.

## Step 2 of 4: right answers

**Does the original system answer new questions right, and which change gets more of them right?**

**[in real life]** In a test before the system goes live, the original system and each change answer the same test questions: the search returns its chunks, and the language model answers from them. The code below writes the 240 answers: 6 versions × 40 questions. `common` saves every answer in a file on this computer, so a re-run reads the answers back instead of writing them again.

In [5]:
answers = []
for name, (way, search, budget, prompt) in SYSTEMS.items():
    for kind in KINDS:
        for n, (_, row) in enumerate(drawn[kind].iterrows(), start=1):
            texts = [index[way]["chunks"][i] for i in retrieve(row["query"], way, search, budget)]
            reply = common.generate(PROMPTS[prompt].format(chunks=numbered(texts), question=row["query"]),
                                    max_tokens=MAX_TOKENS, full=True)
            answers.append({"system": name, "kind": kind, "number": n, "question": row["query"],
                            "correct": row["answer"], "answer": tidy(reply["answer"]),
                            "seconds": reply["prompt_seconds"] + reply["answer_seconds"]})
answers = pd.DataFrame(answers)

**[in real life]** A person reads every answer and rules it *right* when it gives the dataset's correct answer, and *not right* otherwise:
- On a yes/no question, an answer that declines, gives neither yes nor no, or gives both, is *not right*. When a yes/no answer restates the question as a statement instead of writing *yes* or *no*, the hand check reads whether the statement means *yes* or *no* and rules accordingly.
- On a null question, an answer is *right* only when it declines and gives no answer, because no article holds one.
- An answer that gives an answer and then adds *"Not enough information"* is ruled by the answer it gives.
- An answer that leaves out part of the question is *not right*, because it does not answer the whole question.

To keep the code short, the hand check below lists only the answers ruled *not right*, by version, kind and question number (1 to 10 within each kind). Every answer it does not list is *right*.

In [6]:
# Version -> kind -> question number (1-10 within each kind) -> reason. Every answer not listed is ruled right.
NOT_RIGHT = {
    "the original system": {
        "inference": {},
        "comparison": {
            1: "the answer says Engadget suggests no launch and The Verge suggests the shift: a yes, where the correct answer is no",
            4: "the answer says The Verge does not state that Google can address the issues: a no, where the correct answer is yes",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer restates both parts as true: a yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer gives both: inconsistent, then more consistent",
            3: "the answer says the report is consistent: a yes, where the correct answer is no",
            6: "the answer gives neither yes nor no",
            9: "the answer gives neither yes nor no",
        },
        "null": {
            1: "the answer names Rainforest",
            2: "the answer gives the letter C",
            3: "the answer gives the letter M",
            5: "the answer names Kapor Capital",
            6: "the answer names Mikhail Gorbachev",
            8: "the answer gives the letter L",
            9: "the answer names India",
            10: "the answer names Stanford",
        },
    },
    "meaning search": {
        "inference": {
            7: "the answer names Brockman, where the correct answer is Sam Altman",
        },
        "comparison": {
            2: "the answer leaves out the second article, so it does not answer the whole question",
            3: "the answer says Ellison acted independently: a yes, where the correct answer is no",
            4: "the answer says The Verge does not suggest it: a no, where the correct answer is yes",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer restates both parts as true: a yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer says the portrayal was inconsistent: a no, where the correct answer is yes",
            3: "the answer says the report is consistent: a yes, where the correct answer is no",
            6: "the answer gives neither yes nor no",
        },
        "null": {
            1: "the answer names Bolt Financial",
            2: "the answer gives the letter C",
            3: "the answer gives the letter M",
            4: "the answer names Vivek Ramaswamy",
            5: "the answer gives the letter M",
            6: "the answer gives the letter M",
            8: "the answer gives the letter C",
            9: "the answer names Nigeria",
            10: "the answer names Stanford",
        },
    },
    "keyword search": {
        "inference": {
            7: "the answer names Davis, where the correct answer is Sam Altman",
            10: "the answer names Google, where the correct answer is Apple",
        },
        "comparison": {
            3: "the answer says Ellison acted independently: a yes, where the correct answer is no",
            7: "the answer gives neither yes nor no",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer restates both parts as true: a yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer gives neither yes nor no",
            3: "the answer says the report is consistent: a yes, where the correct answer is no",
            6: "the answer says the consistency is undermined: a no, where the correct answer is yes",
            7: "the answer describes an inconsistency: a yes, where the correct answer is no",
            9: "the answer restates all parts as true: a yes, where the correct answer is no",
        },
        "null": {
            2: "the answer gives the letter T",
            3: "the answer gives the letter M",
            5: "the answer names OpenAI",
            6: "the answer gives the letter M",
            8: "the answer gives the letter J",
            9: "the answer names India",
            10: "the answer names Stanford",
        },
    },
    "sentence chunks": {
        "inference": {},
        "comparison": {
            3: "the answer says Ellison acted independently: a yes, where the correct answer is no",
            4: "the answer gives neither yes nor no",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer restates both parts as true: a yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer says the portrayal was inconsistent: a no, where the correct answer is yes",
            6: "the answer gives neither yes nor no",
            7: "the answer gives neither yes nor no",
            8: "the answer gives both: consistent, then not consistently portrayed",
        },
        "null": {
            2: "the answer gives the letter C",
            3: "the answer gives the letter M",
            5: "the answer names Alameda Research",
            6: "the answer names Mikhail Gorbachev",
            8: "the answer names Bret Taylor",
            9: "the answer names Israel",
            10: "the answer names the University of Illinois",
        },
    },
    "1,600 words": {
        "inference": {},
        "comparison": {
            3: "the answer says Ellison acted independently: a yes, where the correct answer is no",
            4: "the answer gives neither yes nor no",
            7: "the answer says The Verge suggests it: a yes, where the correct answer is no",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer restates both parts as true: a yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer says the portrayal was inconsistent: a no, where the correct answer is yes",
            3: "the answer says the portrayal was consistent: a yes, where the correct answer is no",
            6: "the answer gives neither yes nor no",
            7: "the answer gives neither yes nor no",
            8: "the answer gives both: consistent, then more varied",
            9: "the answer restates all parts as true: a yes, where the correct answer is no",
        },
        "null": {
            2: "the answer gives the letter A",
            3: "the answer gives the letter M",
            4: "the answer names Salman Khan",
            5: "the answer gives the letter M",
            6: "the answer gives the letter M",
            7: "the answer names Rohit Sharma",
            8: "the answer gives the letter C",
            9: "the answer names India",
            10: "the answer names Stanford",
        },
    },
    "prompt without rules": {
        "inference": {},
        "comparison": {
            3: "the answer says Ellison acted independently: a yes, where the correct answer is no",
            4: "the answer gives neither yes nor no",
            8: "the answer says the approaches differ: a yes, where the correct answer is no",
            9: "the answer says yes, where the correct answer is no",
        },
        "temporal": {
            2: "the answer gives both: inconsistent, then related",
            3: "the answer declines",
            6: "the answer gives both: consistent, then different details",
            9: "the answer says yes, where the correct answer is no",
        },
        "null": {
            1: "the answer names Rainforest",
            3: "the answer gives the letter M",
            6: "the answer names Mikhail Gorbachev",
            8: "the answer gives the letter B",
            9: "the answer names Japan, India and Francophone Africa",
            10: "the answer names the University of Texas at Austin",
        },
    },
}

answers["hand"] = ["not right" if n in NOT_RIGHT[s][k] else "right"
                   for s, k, n in zip(answers["system"], answers["kind"], answers["number"])]
right = answers[answers["hand"] == "right"]
table = right.pivot_table(index="system", columns="kind", values="number", aggfunc="count", fill_value=0)
table = table.reindex(index=list(SYSTEMS), columns=KINDS, fill_value=0)
rows = [{"version": s, **{k: f"{table.loc[s, k]} of {SAMPLE}" for k in KINDS},
         "total": f"{table.loc[s].sum()} of {SAMPLE * 4}"} for s in SYSTEMS]
print("Right answers, by version and kind:")
print()
show(pd.DataFrame(rows))

Right answers, by version and kind:

version               inference  comparison  temporal  null     total
the original system   10 of 10   6 of 10     6 of 10   2 of 10  24 of 40
meaning search        9 of 10    5 of 10     7 of 10   1 of 10  22 of 40
keyword search        8 of 10    6 of 10     5 of 10   3 of 10  22 of 40
sentence chunks       10 of 10   6 of 10     6 of 10   3 of 10  25 of 40
1,600 words           10 of 10   5 of 10     4 of 10   1 of 10  20 of 40
prompt without rules  10 of 10   6 of 10     6 of 10   4 of 10  26 of 40


**Observation:** the original system gets 24 of the 40 questions right. It gets all 10 inference questions right, and 2 of the 10 null questions: on 8 null questions, the language model gave an answer although no article holds one.

**Observation:** the six versions get from 20 to 26 questions right, all within 4 of the original system's 24.

The output below prints one null question, picked by hand as an example, with its six answers and their rulings, to show what a null question asks and how the six versions answer it. Many null questions ask for a single letter, such as the first letter of a company's name, from facts that no article holds. Every answer in this module is printed word for word, as the language model wrote it.

In [7]:
example = answers[(answers["kind"] == "null") & (answers["number"] == 2)]
print(f"null question 2: {example['question'].iloc[0]}")
print()
for _, row in example.iterrows():
    print(f"{row['system']}, hand check: {row['hand']}")
    print(f"  Answer: {row['answer']}")

null question 2: Considering an article from The Orange County Register discussing the economic impact of a new theme park in Anaheim and a piece from The Los Angeles Times reporting on the latest developments in the city's public transportation system, which letter, representing a transportation project mentioned in both articles, is also the initial of a key attraction in the new theme park?

the original system, hand check: not right
  Answer: The letter "C" is the initial of a key attraction in the new theme park, as it is mentioned in the context of a "Cruise" drama in California. [2] The letter "C" is also the initial of a key attraction in the new theme park, as it is mentioned in the context of a "Cruise" drama in California, and the Cruise is a key attraction in the new theme park. [2] Not enough information.
meaning search, hand check: not right
  Answer: The initial of the key attraction in the new theme park is "C", which is mentioned in the context of the Cruise drama unfo

**Observation:** five of the six versions give a letter, each citing a chunk, although no article holds the answer. Only the prompt without rules declines. Over all 10 null questions, the original system declined on 2 and the prompt without rules on 4: rule 3, which asks the language model to decline, did not make it decline more here. On 10 questions, a gap of 2 is too small to tell apart from chance.

Totals alone hide how many rulings changed: a change can turn some answers right and others not right, and the two cancel in the total. The table below counts both, question by question, and tests the gap with the same 95% interval as Step 1. Here each question's difference is 1 when its answer turned right, −1 when it turned not right, and 0 when its ruling stayed the same. The two counts set the interval's width: the more rulings a change turns both ways, the larger the standard deviation of the differences, and the wider the interval around the same gap. The gap and its interval are multiplied by 40, so that they read in answers.

In [8]:
is_right = lambda name: (answers[answers["system"] == name].set_index(["kind", "number"])["hand"] == "right").astype(int)
whole = lambda v: f"{v:+.1f}".replace("-", "−")
answer_gaps, rows = {}, []
for name in list(SYSTEMS)[1:]:
    differences = is_right(name) - is_right("the original system")          # 1 = turned right, −1 = turned not right, 0 = same ruling
    gap, low, high = (SAMPLE * 4 * v for v in gap_and_interval(differences))
    answer_gaps[name] = (gap, low > 0 or high < 0)
    rows.append({"change": name, "turned right": str((differences == 1).sum()), "turned not right": str((differences == -1).sum()),
                 "gap in right answers": f"{gap:+.0f}".replace("-", "−"),
                 "95% interval of the gap": f"{whole(low)} to {whole(high)}",
                 "real gap": "yes" if answer_gaps[name][1] else "no"})
print(f"Rulings that changed from the original system's, on the same {SAMPLE * 4} questions:")
print()
show(pd.DataFrame(rows))

Rulings that changed from the original system's, on the same 40 questions:

change                turned right  turned not right  gap in right answers  95% interval of the gap  real gap
meaning search                   2                 4                    −2  −6.8 to +2.8             no
keyword search                   3                 5                    −2  −7.6 to +3.6             no
sentence chunks                  4                 3                    +1  −4.2 to +6.2             no
1,600 words                      2                 6                    −4  −9.5 to +1.5             no
prompt without rules             3                 1                    +2  −1.9 to +5.9             no


**Observation:** every change turned some answers right and others not right. 1,600 words and keyword search changed the most rulings, 8 each. 1,600 words turned 2 right and 6 not right, a gap of −4.

**Observation:** no gap in right answers is real: 0 is inside every interval. On 40 questions, a gap of a few answers can come from chance: another 40 questions could turn a different few. The doubled interval of Step 1 is not added here: it only widens an interval, and every interval here already holds 0.

**Conclusion:** the original system answers 24 of the 40 new questions right, and no change to one setting gets more of them right by more than chance.

## Step 3 of 4: one table with recall gaps, right-answer gaps, and time per answer

**Which setting moved the results most?**

Steps 1 and 2 graded the same changes two ways: Step 1 graded the search alone, on 648 questions, and Step 2 graded the answers, on 40. The table below puts each change's two gaps side by side, with a third result: the seconds per answer, the time the language model took to read the prompt and write the answer, averaged over the 40 questions. Time matters because a change that makes every answer slower has to add more right answers to be worth it.

In [9]:
seconds = answers.groupby("system")["seconds"].mean()
rows = []
for name in list(SYSTEMS)[1:]:
    gap, low, high = gap_and_interval(recalls[name] - recalls["the original system"])
    real = low > 0 or high < 0
    answer_gap, answer_real = answer_gaps[name]
    rows.append({"change": name, "part": PART[name],
                 "recall gap": "—" if name == "prompt without rules" else sign(gap) + (" (real)" if real else " (not real)"),
                 "gap in right answers": f"{answer_gap:+.0f}".replace("-", "−") + (" (real)" if answer_real else " (not real)"),
                 "seconds per answer": f"{seconds[name]:.1f}",
                 "time gap: % of the original system's": f"{(seconds[name] - seconds['the original system']) / seconds['the original system']:+.0%}".replace("-", "−")})
print(f"Seconds per answer, the original system = {seconds['the original system']:.1f}")
print()
print("Each change's gaps from the original system:")
print()
show(pd.DataFrame(rows))

Seconds per answer, the original system = 15.8

Each change's gaps from the original system:

change                part         recall gap     gap in right answers  seconds per answer  time gap: % of the original system's
meaning search        4, retrieve  −0.093 (real)  −2 (not real)                       15.4                                   −2%
keyword search        4, retrieve  +0.043 (real)  −2 (not real)                       18.6                                  +18%
sentence chunks       1, chunk     +0.074 (real)  +1 (not real)                       19.8                                  +25%
1,600 words           4, retrieve  +0.254 (real)  −4 (not real)                       52.6                                 +233%
prompt without rules  5, generate  —              +2 (not real)                       16.2                                   +3%


**Observation:** 1,600 words moved the search grade the most, by a real gap. In right answers, its gap was the largest, 4 fewer, but not real. It also made each answer slower, because the language model reads four times as much text: time per answer, 1,600 words / the original system = 52.6 / 15.8 = 3.3.

**Observation:** a real gain in recall did not show up as a real gain in right answers. Keyword search raised recall by a real gap, and got 2 fewer answers right. Sentence chunks raised recall too, and got 1 more right. On 40 questions, neither answer gap is real.

**Observation:** with the same limit of 400 words, keyword search and sentence chunks took a few seconds longer per answer than the original system. The seconds also depend on how long each answer is, which differs between versions, so a gap of a few seconds is not a cost of the change alone.

**Hypothesis:** better recall added no right answers because many of the original system's failures sit where the search cannot help. The original system has 40 − 24 = 16 not-right answers, and 8 of them are on null questions, where no chunk holds an answer, so only the language model can decline. This module does not trace the other 8 failures to the search or the language model.

**Conclusion:** the setting that moved the search grade most was the words returned. No setting moved the right answers by more than chance, so a change that improves the search grade still has to be tested on the answers before it is kept.

## Step 4 of 4: the guide

**Given a new problem, which tool fits it?**

Every problem below asks a language model for something it does not already know or do. The guide matches the shape of the problem to the tool, with the reason for each:

| the problem's shape | the tool that fits | why |
|---|---|---|
| The answer is a fact the language model never read, held in a few passages of a large collection that keeps changing | RAG | The search hands the language model only the chunks that match the question. A new document needs only its chunks added to the store, and each answer cites chunks a reader can check. |
| The collection is small enough to fit in the language model's context window, the most text it can read at once | Paste the whole collection into the prompt | With every document in the prompt, no search can miss a fact. But the language model reads the whole collection for every question, so this stops working as the collection grows. |
| The answer is a count, a total or an average over the whole collection | A database query, such as SQL, over a table of the documents, one row per document | The search returns a few hundred words, so the language model sees only a few documents. A query reads every row. |
| The answer may be in no document of the collection | RAG, tested on questions that have no answer before it is trusted | The search always returns chunks, even when none holds the answer, so only the language model can decline. In this module, the original system declined on 2 of 10 such questions, and the prompt without rules on 4 of 10: neither declines reliably. |
| The need is a new skill, such as a kind of task the language model does poorly, not new facts | Retraining the language model, called fine-tuning | RAG adds facts to the prompt. It does not train the language model, so the language model's skills stay what they were. |
| A RAG system whose answers are often not right | Trace each failure to the search or the language model, then change the part that failed | In this module, every change to the chunks or the search raised or lowered recall by a real gap, but none changed the right answers by more than chance. So a better search grade does not show better answers, and the failing part has to be found first. |

## Key takeaways
- A system is tested by changing one setting at a time and grading each change on the same questions, so that every gap comes from that one setting.
- Two grades answer two questions. The search grade (recall) grades the returned chunks, so it needs no answers and can run on hundreds of questions. Right answers grade the whole system, but every answer has to be written by the language model and then ruled by reading it, so they run on fewer questions.
- On these questions, the words returned moved the search grade most. Sentence chunks and keyword search raised it less, and meaning search lowered it.
- No change moved right answers by more than chance on 40 questions. The change that raised recall most, 1,600 words, made each answer 3.3 times as slow, for no real gain in right answers.
- A gap is read with its interval: a gap of a few answers out of 40 can come from chance.
- Half of the original system's not-right answers were on questions with no answer, where only the language model can decline. That may be why better recall added no right answers, a hypothesis this module does not test.
- These results are rough, for three reasons:
  - 10 questions per kind is small;
  - a language model, not a person, made the hand check;
  - each change was tried alone, so two changes together could act differently.

## Conclusion
A better search grade does not by itself give more right answers: each change has to be graded on the answers too, and the right tool for a problem, RAG or another, depends on what the problem asks for, as the guide shows.